# Ingestión Manual de Datos (PDFs y Requisitos)
Este cuaderno permite ingestar directamente a la base de datos (PostgreSQL/pgvector) los requerimientos etiquetados del dataset de prueba y procesar la norma ISO 29148 desde su PDF, dividiéndola en fragmentos (chunks) de texto de tamaño configurado.

## 1. Configuración Inicial

In [ ]:
import sys
import os
import pandas as pd
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

# Añadir el directorio backend al path para poder importar los servicios
sys.path.append(os.path.abspath('../backend'))

load_dotenv('../.env')

from app.services.retriever.retriever_service import index_requirements, index_documents
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pdfplumber

## 2. Ingesta de Requerimientos desde CSV
Cada requerimiento se almacena como un único fragmento en la tabla `requirements`.

In [ ]:
csv_path = '../data/raw/requirements.csv'

print("Cargando dataset de requerimientos...")
df_reqs = pd.read_csv(csv_path)

# Limpiamos nulos y obtenemos la lista de textos
req_texts = df_reqs['requirement'].dropna().tolist()
print(f"Total de requerimientos a indexar: {len(req_texts)}")

# Indexamos los requisitos (esto generará los embeddings según el proveedor configurado)
# ADVERTENCIA: Puede tomar un tiempo dependiendo de si usas LLM local o API
indexed_count = index_requirements(
    texts=req_texts,
    source="csv",
    source_name="requirements.csv"
)

print(f"Requerimientos indexados exitosamente: {indexed_count}")

## 3. Preprocesamiento e Ingesta del PDF (Norma ISO)
Se lee el texto del PDF, se elimina ruido (espacios, saltos de página) y se divide en fragmentos de **400 caracteres con 100 de solapamiento**.

In [ ]:
pdf_path = '../data/raw/iso_29148.pdf'

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

print("Extrayendo texto del PDF de la norma ISO...")
if os.path.exists(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    print(f"Caracteres extraídos: {len(raw_text)}")
    
    # Configurar el text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Crear chunks
    chunks = text_splitter.split_text(raw_text)
    print(f"Se generaron {len(chunks)} chunks.")
    
    # Convertir a objetos Document de LangChain
    documents = [
        Document(
            page_content=chunk, 
            metadata={"source": "pdf", "source_file": "iso_29148.pdf", "chunk_index": i}
        ) 
        for i, chunk in enumerate(chunks)
    ]
    
    # Indexar documentos en la base de datos vectorial
    print("Indexando chunks en la base de datos...")
    docs_indexed = index_documents(documents)
    print(f"Chunks indexados exitosamente: {docs_indexed}")
else:
    print(f"No se encontró el archivo: {pdf_path}")